# Encoder arm v2: standardised fold scores + collapse restarts

Runtime → Change runtime type → **T4 GPU** (or L4). Then Runtime → Run all.

Changes from v1: each fold model's scores are standardised by its own out-of-fold mean and SD before pooling, and a fine-tune whose outputs collapse is restarted with a new seed (up to 2 times).

Results go to a fresh Drive folder, `MyDrive/capstone_runs_v2/`, so they never mix with the v1 run.

In [5]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
REPO = 'https://github.com/ROLFFFX/ehr-inbox-triage.git'
%cd /content
!rm -rf ehr-inbox-triage && git clone -q $REPO
%cd /content/ehr-inbox-triage
!git log --oneline -1
!grep -q COLLAPSE_SD triage/methods/encoder.py && echo 'v2 code OK' || echo 'STOP: the repo does not have the v2 code yet - push it first'
!pip install -q -r requirements.txt -r requirements-gpu.txt

/content
/content/ehr-inbox-triage
7ca9c4d (HEAD -> main, origin/main, origin/HEAD) standardize fold scores, restart collapsed fine tuning on bert
v2 code OK


In [8]:
# Keep results on Drive in a fresh v2 folder, seeded with the repo's (v2) baseline CSVs.
RUNS = '/content/drive/MyDrive/capstone_runs_v2'
!mkdir -p $RUNS/results
!rsync -a --ignore-existing --exclude archive results/ $RUNS/results/
!rm -rf results && ln -s $RUNS/results results
!ls results

figures  primary  sens_level2


In [9]:
!python -m scripts.download_data

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/751k [00:00<?, ?B/s]          
Reconstructing (incomplete total...):   0% 0.00/751k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/752k [00:00<?, ?B/s]

Fetching 4 files:  25% 1/4 [00:00<00:00,  3.85it/s]
Reconstructing (incomplete total...):   0% 465/754k [00:00<06:54, 1.82kB/s]

Fetching 4 files:  75% 3/4 [00:01<00:00,  2.22it/s]
Fetching 4 files: 100% 4/4 [00:01<00:00,  3.00it/s]
Download complete: 100% 750k/750k [00:01<00:00, 610kB/s,  610kB/s  ]
Reconstruction complete: 100% 754k/754k [00:01<00:00, 614kB/s,  614kB/s  ]             PortalPal-AI/PMR-Reddit-Train-Dev-Pool -> /content/ehr-inbox-triage/data/raw/PMR-Reddit-Train-Dev-Pool




Reconstructing (incomple

## 1. Quick check (~1 min): tiny model, one epoch
The tiny model usually collapses, so expect a few "restarting with a new seed" lines. That is the restart logic working. If this cell errors, stop and send me the error.

In [10]:
!python -m scripts.run_encoders --models tiny --heads binary --epochs 1 --max-len 128 --n-boot 50 --tag smoke

                               n  n_urgent  prevalence  median_chars
source             split                                            
pmr_reddit         train     772        78       0.101           773
                   dev        87         9       0.103           778
                   test      362        38       0.105           768
pmr_synth          INBOX_A    30         5       0.167           389
                   INBOX_B    30         5       0.167           404
cssrs_suicidewatch all      1170       245       0.209           283 

config.json: 100% 382/382 [00:00<00:00, 1.69MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 24.1MB/s]

model.safetensors: downloading bytes:   0% 0.00/17.7M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  94% 16.7M/17.7M [00:00<00:00, 30.4MB/s,  622kB/s  ]
model.safetensors: downloading bytes: 100% 16.7M/16.7M [00:00<00:00, 18.6MB/s, 1.63MB/s  ]
model.safetensors: reconstructing file: 100% 17.7M/17.7M [00:00<00:00, 19.8MB/s, 1.73MB/s  ]


## 2. Main run, seed 0 (~35 min on a T4)

In [11]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed 0 --tag primary

                               n  n_urgent  prevalence  median_chars
source             split                                            
pmr_reddit         train     772        78       0.101           773
                   dev        87         9       0.103           778
                   test      362        38       0.105           768
pmr_synth          INBOX_A    30         5       0.167           389
                   INBOX_B    30         5       0.167           404
cssrs_suicidewatch all      1170       245       0.209           283 

config.json: 100% 385/385 [00:00<00:00, 2.10MB/s]
vocab.txt: 100% 213k/213k [00:00<00:00, 23.1MB/s]

pytorch_model.bin: downloading bytes:  56% 242M/436M [00:01<00:00, 336MB/s, 18.9MB/s  ]
pytorch_model.bin: downloading bytes:  85% 371M/436M [00:01<00:00, 328MB/s, 31.7MB/s  ]
pytorch_model.bin: downloading bytes:  94% 412M/436M [00:01<00:00, 241MB/s, 36.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 412M/412M [00:04<00:00, 94.8MB/s, 37.0

## 3. Extra seeds for training variance (~70 min)

In [12]:
for s in (1, 2):
    !python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --model-seed {s} --tag seeds/s{s}

                               n  n_urgent  prevalence  median_chars
source             split                                            
pmr_reddit         train     772        78       0.101           773
                   dev        87         9       0.103           778
                   test      362        38       0.105           768
pmr_synth          INBOX_A    30         5       0.167           389
                   INBOX_B    30         5       0.167           404
cssrs_suicidewatch all      1170       245       0.209           283 

Loading weights: 100% 199/199 [00:00<00:00, 15730.02it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |

## 4. Sensitivity definition (Levels 1–2), seed 0 (~35 min)

In [13]:
!python -m scripts.run_encoders --models bioclinicalbert roberta --heads binary ordinal --max-level 2 --tag sens_level2

                               n  n_urgent  prevalence  median_chars
source             split                                            
pmr_reddit         train     772       108       0.140           773
                   dev        87        12       0.138           778
                   test      362        51       0.141           768
pmr_synth          INBOX_A    30        10       0.333           389
                   INBOX_B    30        10       0.333           404
cssrs_suicidewatch all      1170       245       0.209           283 

Loading weights: 100% 199/199 [00:00<00:00, 16162.24it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED |

## 5. Summary and bundle
Download `encoder_results_v2.zip` from `MyDrive/capstone_runs_v2/`.

In [14]:
import pandas as pd
for t in ['primary', 'seeds/s1', 'seeds/s2', 'sens_level2']:
    try:
        c = pd.read_csv(f'{RUNS}/results/{t}/classification.csv')
        c = c[c.method.str.startswith(('bioclinicalbert', 'roberta'))]
        print('==', t); print(c[['method', 'recall', 'fa_per_100', 'auroc', 'recall_at_ppv67']].round(3).to_string(index=False))
    except FileNotFoundError:
        print('==', t, 'missing')

== primary
                 method  recall  fa_per_100  auroc  recall_at_ppv67
 bioclinicalbert_binary   0.974      87.654  0.741            0.184
bioclinicalbert_ordinal   0.974      88.889  0.753            0.053
         roberta_binary   1.000      76.852  0.803            0.053
        roberta_ordinal   0.974      78.704  0.814            0.000
== seeds/s1
                 method  recall  fa_per_100  auroc  recall_at_ppv67
 bioclinicalbert_binary   1.000      78.704  0.795            0.053
bioclinicalbert_ordinal   1.000      90.741  0.742            0.000
         roberta_binary   0.947      66.667  0.824            0.053
        roberta_ordinal   1.000      83.642  0.829            0.079
== seeds/s2
                 method  recall  fa_per_100  auroc  recall_at_ppv67
 bioclinicalbert_binary   0.974      89.198  0.754            0.053
bioclinicalbert_ordinal   1.000      85.494  0.704            0.079
         roberta_binary   1.000      73.148  0.815            0.053
        rober

In [15]:
!cd $RUNS && zip -qr encoder_results_v2.zip results -x '*/smoke/*' && ls -lh $RUNS/encoder_results_v2.zip

-rw------- 1 root root 14M Nov 13 05:55 /content/drive/MyDrive/capstone_runs_v2/encoder_results_v2.zip


In [16]:
!pip list 2>/dev/null | grep -iE
"^(torch|transformers|accelerate|scikit-learn|numpy|pandas) "
!python --version


Usage: grep [OPTION]... PATTERNS [FILE]...
Try 'grep --help' for more information.
Python 3.13.16
